### Notebook collecting commands/scripts which are run after fits/scans complete

In [1]:
# imports
# autoreload magic:
%load_ext autoreload
%autoreload 2
import glob
import copy
import matplotlib.pyplot as plt
import numpy as np
import os
import glob
from scipy.stats import norm
%matplotlib inline
import sys

from NNMFit.utilities import load_pickle
from NNMFit.utilities import ScanHandler

from freefit_parameter_config import FreefitParamConfig

run scan analysis: calculate MC Histogram, Data Histogram, and Saturated LLH of the best fit

In [2]:
all_step0_scans = glob.glob(
    "//data/user/tvaneede/GlobalFit/reco_processing/NNMFit/dag_scans/flavor_globalfit/hese/unblind/step1_hese_flux/data_fits/force_full_3_nlight_20/nbestfit20_scan_astro_gamma_1D_10steps_round2"
)
all_step0_scans

['//data/user/tvaneede/GlobalFit/reco_processing/NNMFit/dag_scans/flavor_globalfit/hese/unblind/step1_hese_flux/data_fits/force_full_3_nlight_20/nbestfit20_scan_astro_gamma_1D_10steps_round2']

In [5]:
for scan_dir in all_step0_scans:
    # run do_scan_analysis.py with scan_dir as arg:
    print(f"Processing scan directory: {scan_dir}")
    os.system(f"python do_scan_analysis.py {scan_dir}")

Processing scan directory: //data/user/tvaneede/GlobalFit/reco_processing/NNMFit/dag_scans/flavor_globalfit/hese/unblind/step1_hese_flux/data_fits/force_full_3_nlight_20/nbestfit20_scan_astro_gamma_1D_10steps_round2
Auto-Detecting the best fit from among the freefits in the scan path.
Initializing on server cobalt-14.icecube.wisc.edu. 
Using config directory: /data/user/tvaneede/GlobalFit/reco_processing/NNMFit/configs/flavor_globalfit/analysis_configs/input_pars_only
found the minimum llh value:  212.8606063267837
Found 20 freefit files in //data/user/tvaneede/GlobalFit/reco_processing/NNMFit/dag_scans/flavor_globalfit/hese/unblind/step1_hese_flux/data_fits/force_full_3_nlight_20/nbestfit20_scan_astro_gamma_1D_10steps_round2
This is the index of the best fit: 17, the name is //data/user/tvaneede/GlobalFit/reco_processing/NNMFit/dag_scans/flavor_globalfit/hese/unblind/step1_hese_flux/data_fits/force_full_3_nlight_20/nbestfit20_scan_astro_gamma_1D_10steps_round2/Freefit_17.pickle
found 

/home/pfuerst/software/icecube/NNMFit/NNMFit/utilities/result_handlers.py:201: PerformanceWarning: 
your performance may suffer as PyTables will pickle object types that it cannot
map directly to c-types [inferred_type->mixed,key->block0_values] [items->Index(['CR_grad', 'astro_norm', 'barr_h', 'barr_w', 'barr_y', 'barr_z',
       'conv_norm', 'delta_gamma', 'dom_eff', 'fit_success', 'gamma_astro',
       'ice_abs', 'ice_crystal', 'ice_holep0', 'ice_holep1', 'ice_scat', 'llh',
       'muongun_norm', 'prompt_norm'],
      dtype='object')]

  df.to_hdf(self.scan_result_file, key="scans")


Data Histogram already exists: //data/user/tvaneede/GlobalFit/reco_processing/NNMFit/dag_scans/flavor_globalfit/hese/unblind/step1_hese_flux/data_fits/force_full_3_nlight_20/nbestfit20_scan_astro_gamma_1D_10steps_round2/Data_Histogram.pickle
Loading Data Histogram: //data/user/tvaneede/GlobalFit/reco_processing/NNMFit/dag_scans/flavor_globalfit/hese/unblind/step1_hese_flux/data_fits/force_full_3_nlight_20/nbestfit20_scan_astro_gamma_1D_10steps_round2/Data_Histogram.pickle
LLH Map already exists: //data/user/tvaneede/GlobalFit/reco_processing/NNMFit/dag_scans/flavor_globalfit/hese/unblind/step1_hese_flux/data_fits/force_full_3_nlight_20/nbestfit20_scan_astro_gamma_1D_10steps_round2/LLH_Maps_Freefit_17.pickle
Saturated LLH: 90.96186371833632
Min LLH: 212.8606063267837, sat. LLH: 90.96186371833632
LLH info saved to: /data/user/tvaneede/GlobalFit/reco_processing/NNMFit/notebooks/unblind/philipp/saturated/force_full_3_nlight_20/SatLLH_nbestfit20_scan_astro_gamma_1D_10steps_round2.txt


Collect the bestfit parameter values in a config file (for later pseudoexperiments etc.)

In [6]:
# get the "names" of the scans, i.e. the last dirname in the path
scan_names = [os.path.basename(path) for path in all_step0_scans]
scan_names

['nbestfit20_scan_astro_gamma_1D_10steps_round2']

In [8]:
# find the best-fit for each scan and write a configuration file using the
# bestfit params as input params:
freefit_config_names = {}
min_llh_dict = {}
for scan_path in all_step0_scans:
    print(scan_path)
    freefit_config_hdl = FreefitParamConfig(scan_path)
    print(freefit_config_hdl.get_name())
    freefit_config_hdl.write()
    min_llh_dict[freefit_config_hdl.get_name()
                ] = freefit_config_hdl.get_min_llh_freefit(
                    freefit_config_hdl.scan_hdl
                )
    # add the config name to the dictionary:
    freefit_config_names[os.path.basename(os.path.dirname(scan_path))
                        ] = freefit_config_hdl.get_name()

//data/user/tvaneede/GlobalFit/reco_processing/NNMFit/dag_scans/flavor_globalfit/hese/unblind/step1_hese_flux/data_fits/force_full_3_nlight_20/nbestfit20_scan_astro_gamma_1D_10steps_round2
Initializing on server cobalt-14.icecube.wisc.edu. 
Using config directory: /data/user/tvaneede/GlobalFit/reco_processing/NNMFit/configs/flavor_globalfit/analysis_configs/input_pars_only
found the minimum llh value:  212.8606063267837
Found 20 freefit files in //data/user/tvaneede/GlobalFit/reco_processing/NNMFit/dag_scans/flavor_globalfit/hese/unblind/step1_hese_flux/data_fits/force_full_3_nlight_20/nbestfit20_scan_astro_gamma_1D_10steps_round2
This is the index of the best fit: 17, the name is //data/user/tvaneede/GlobalFit/reco_processing/NNMFit/dag_scans/flavor_globalfit/hese/unblind/step1_hese_flux/data_fits/force_full_3_nlight_20/nbestfit20_scan_astro_gamma_1D_10steps_round2/Freefit_17.pickle
/data/user/tvaneede/GlobalFit/reco_processing/NNMFit/configs/flavor_globalfit/analysis_configs/input_pa

/home/pfuerst/software/icecube/NNMFit/NNMFit/utilities/result_handlers.py:201: PerformanceWarning: 
your performance may suffer as PyTables will pickle object types that it cannot
map directly to c-types [inferred_type->mixed,key->block0_values] [items->Index(['CR_grad', 'astro_norm', 'barr_h', 'barr_w', 'barr_y', 'barr_z',
       'conv_norm', 'delta_gamma', 'dom_eff', 'fit_success', 'gamma_astro',
       'ice_abs', 'ice_crystal', 'ice_holep0', 'ice_holep1', 'ice_scat', 'llh',
       'muongun_norm', 'prompt_norm'],
      dtype='object')]

  df.to_hdf(self.scan_result_file, key="scans")
